<a href="https://colab.research.google.com/github/crisss330-svg/Pratica1_Programacao_Drivers/blob/main/comunicacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""
Created on Sat Oct 31 11:22:55 2026

@author: cristiano_oldenburg

Pratica 1 programação de drive
"""

import socket
from umodbus.client import tcp

# Constantes do dispositivo
SLAVE_ID = 2
PORT = 502

# Endereços dos registradores
REG_MOTOR_ON_OFF = 1100
REG_MOTOR_DIR = 1101
REG_MOTOR_STATE = 100
REG_SPEED_READ = 30400
REG_CURRENT = 30401
REG_VOLTAGE = 30402
REG_TEMP = 30403
REG_SPEED_WRITE = 41400

def conectar():
    """Solicita o IP ao usuário (professor) e estabelece a conexão TCP."""
    ip = input("Digite o endereço IP do inversor (fornecido pelo professor): ")
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        sock.connect((ip, PORT))
        print(f"Conectado ao inversor no IP {ip}")
        return sock
    except Exception as e:
        print(f"Erro ao conectar: {e}")
        return None

def enviar_comando(sock, message):
    """Função auxiliar para enviar a mensagem Modbus e retornar a resposta."""
    try:
        return tcp.send_message(message, sock)
    except Exception as e:
        print(f"Erro de comunicação: {e}")
        return None

def iniciar_padrao(sock):
    """Inicia o motor com 30Hz e sentido horário."""
    # Sentido horário (0)
    msg_dir = tcp.write_single_coil(slave_id=SLAVE_ID, address=REG_MOTOR_DIR, value=0)
    enviar_comando(sock, msg_dir)

    # Velocidade 30Hz
    msg_speed = tcp.write_single_register(slave_id=SLAVE_ID, address=REG_SPEED_WRITE, value=30)
    enviar_comando(sock, msg_speed)

    # Ligar motor (1)
    msg_on = tcp.write_single_coil(slave_id=SLAVE_ID, address=REG_MOTOR_ON_OFF, value=1)
    enviar_comando(sock, msg_on)

    print("Motor iniciado com configuração padrão (30Hz, Sentido Horário).")

def menu():
    print("\n--- PAINEL DE CONTROLE DO INVERSOR SENAI ---")
    print("1. Ligar motor")
    print("2. Parar motor")
    print("3. Definir velocidade (1 a 60Hz)")
    print("4. Mudar sentido de giro")
    print("5. Verificar estado do motor (Girando/Parado)")
    print("6. Verificar velocidade atual")
    print("7. Verificar temperatura")
    print("8. Verificar corrente")
    print("9. Verificar tensão")
    print("10. Iniciar com Configuração Padrão (30Hz, Horário)")
    print("0. Sair")
    return input("Escolha uma opção: ")

def main():
    sock = conectar()
    if not sock:
        return

    while True:
        opcao = menu()

        if opcao == '1':
            msg = tcp.write_single_coil(slave_id=SLAVE_ID, address=REG_MOTOR_ON_OFF, value=1)
            enviar_comando(sock, msg)
            print("Comando para LIGAR enviado.")

        elif opcao == '2':
            msg = tcp.write_single_coil(slave_id=SLAVE_ID, address=REG_MOTOR_ON_OFF, value=0)
            enviar_comando(sock, msg)
            print("Comando para PARAR enviado.")

        elif opcao == '3':
            try:
                vel = int(input("Digite a velocidade desejada (1 a 60): "))
                if 1 <= vel <= 60:
                    msg = tcp.write_single_register(slave_id=SLAVE_ID, address=REG_SPEED_WRITE, value=vel)
                    enviar_comando(sock, msg)
                    print(f"Velocidade definida para {vel}Hz.")
                else:
                    print("Erro: A velocidade deve estar entre 1 e 60 Hz.")
            except ValueError:
                print("Entrada inválida.")

        elif opcao == '4':
            print("0 - Horário | 1 - Anti-horário")
            dir_val = int(input("Escolha o sentido: "))
            if dir_val in [0, 1]:
                msg = tcp.write_single_coil(slave_id=SLAVE_ID, address=REG_MOTOR_DIR, value=dir_val)
                enviar_comando(sock, msg)
                sentido = "Anti-horário" if dir_val == 1 else "Horário"
                print(f"Sentido alterado para {sentido}.")
            else:
                print("Valor inválido.")

        elif opcao == '5':
            msg = tcp.read_discrete_inputs(slave_id=SLAVE_ID, starting_address=REG_MOTOR_STATE, quantity=1)
            resp = enviar_comando(sock, msg)
            if resp:
                estado = "Girando" if resp[0] == 1 else "Parado"
                print(f"Estado do Motor: {estado}")

        elif opcao == '6':
            msg = tcp.read_input_registers(slave_id=SLAVE_ID, starting_address=REG_SPEED_READ, quantity=1)
            resp = enviar_comando(sock, msg)
            if resp: print(f"Velocidade atual: {resp[0]} Hz")

        elif opcao == '7':
            msg = tcp.read_input_registers(slave_id=SLAVE_ID, starting_address=REG_TEMP, quantity=1)
            resp = enviar_comando(sock, msg)
            if resp: print(f"Temperatura: {resp[0]} °C")

        elif opcao == '8':
            msg = tcp.read_input_registers(slave_id=SLAVE_ID, starting_address=REG_CURRENT, quantity=1)
            resp = enviar_comando(sock, msg)
            if resp: print(f"Corrente: {resp[0]} A")

        elif opcao == '9':
            msg = tcp.read_input_registers(slave_id=SLAVE_ID, starting_address=REG_VOLTAGE, quantity=1)
            resp = enviar_comando(sock, msg)
            if resp: print(f"Tensão: {resp[0]} V")

        elif opcao == '10':
            iniciar_padrao(sock)

        elif opcao == '0':
            print("Encerrando programa...")
            sock.close()
            break
        else:
            print("Opção inválida.")

if __name__ == "__main__":
    main()


KeyboardInterrupt: Interrupted by user

In [ ]:
#!/usr/bin/env python
# scripts/examples/simple_tcp_client.py
import socket

from umodbus import conf
from umodbus.client import tcp

# Enable values to be signed (default is False).
conf.SIGNED_VALUES = True

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.connect(('localhost', 502))

# Returns a message or Application Data Unit (ADU) specific for doing
# Modbus TCP/IP.
message = tcp.write_multiple_coils(slave_id=1, starting_address=1, values=[1, 0, 1, 1])

# Response depends on Modbus function code. This particular returns the
# amount of coils written, in this case it is.
response = tcp.send_message(message, sock)

sock.close()